## Feasibility of Enacted Maps to Inexact Contiguity Constraints (Hop)

Checks whether each district in an enacted plan is feasible under tree-based, distance-based, and DAG-based contiguity constraints using **unweighted (hop)** shortest paths.

### Data Requirements

1. **Block-level graphs** — stored in `../../data/{state}_block.json`
2. **Block Assignment Files (BAFs)** — run `bash scripts/download_baf.sh` to download and extract, or manually download from:
   - Congressional: https://www2.census.gov/programs-surveys/decennial/rdo/mapping-files/2023/118-congressional-district-bef/cd118.zip
   - State Senate: https://www2.census.gov/programs-surveys/decennial/rdo/mapping-files/2023/2022-state-legislative-bef/sldu_2022.zip
   - State House: https://www2.census.gov/programs-surveys/decennial/rdo/mapping-files/2023/2022-state-legislative-bef/sldl_2022.zip

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT_DIR))

from src.read import read_graph_from_json
from src.params import BaseParams
from src.utils import nearest_node
import networkx as nx
import pandas as pd

In [3]:
# Paths to data
filepath_graphs = str(ROOT_DIR / "data") + "/"
filepath_cd = str(ROOT_DIR / "data" / "baf" / "cd118") + "/"
filepath_ss = str(ROOT_DIR / "data" / "baf" / "sldu_2022") + "/"
filepath_sh = str(ROOT_DIR / "data" / "baf" / "sldl_2022") + "/"

In [4]:
def get_fips(state):
    return BaseParams.state_fips_mapping[state]


# Build number_of_districts from params for all states
number_of_districts = {}
for state, k in BaseParams.congressional_districts_2020.items():
    number_of_districts[(state, "CD")] = k
for state, k in BaseParams.state_senate_districts_2020.items():
    number_of_districts[(state, "SS")] = k
for state, k in BaseParams.state_house_districts_2020.items():
    number_of_districts[(state, "SH")] = k

print(f"Total (state, district_type) pairs: {len(number_of_districts)}")

Total (state, district_type) pairs: 150


In [5]:
def read_enacted_districts(G, state, district_type):
    """Read enacted district assignments from a Census BAF file."""
    geoid_to_node = {G.nodes[i]['GEOID20']: i for i in G.nodes}

    if district_type == 'CD':
        fx = '_CD118.txt'
        filepath = filepath_cd
        col = 'CDFP'
    elif district_type == 'SS':
        fx = '_SLDU22.txt'
        filepath = filepath_ss
        col = 'SLDUST'
    elif district_type == 'SH':
        fx = '_SLDL22.txt'
        filepath = filepath_sh
        col = 'SLDLST'
    else:
        raise ValueError(f"Unknown district_type: {district_type}")

    fips = get_fips(state)
    filename = fips + '_' + state + fx
    csv_file = pd.read_csv(filepath + filename, skipinitialspace=True)

    districts = {}
    unassigned = []

    for _, row in csv_file.iterrows():
        g = str(row['GEOID'])
        if len(g) < 15:  # fix leading zeros
            g = '0' + g

        i = geoid_to_node[g]
        j = str(row[col])

        if j in {'ZZ', 'ZZZ'}:
            unassigned.append(i)
        else:
            if j not in districts:
                districts[j] = []
            districts[j].append(i)

    if unassigned:
        print(f"  unassigned = {unassigned}")
    return districts

In [6]:
import os

counts = {"ttt": 0, "ftt": 0, "fft": 0, "fff": 0}

for (state, district_type) in number_of_districts.keys():

    # Skip if block graph file doesn't exist
    graph_file = filepath_graphs + state + "_block.json"
    if not os.path.exists(graph_file):
        continue

    print(f"{'*'*40}")
    print(f"Starting {state} {district_type}")
    print(f"{'*'*40}\n")

    # Read block-level graph (use cache to avoid re-reading)
    G = read_graph_from_json(graph_file, state=state)
    print(f"  nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}")

    if number_of_districts[state, district_type] <= 1:
        print("  Skipping because k <= 1.")
        continue
    if not nx.is_connected(G):
        print("  Skipping because G is disconnected.")
        continue

    # Read enacted districts from BAF
    districts = read_enacted_districts(G, state, district_type)

    for key in districts:
        district = districts[key]
        population = sum(G.nodes[i]['TOTPOP'] for i in district)
        mean_x = sum(G.nodes[i]['TOTPOP'] * G.nodes[i]['C_X'] for i in district) / population
        mean_y = sum(G.nodes[i]['TOTPOP'] * G.nodes[i]['C_Y'] for i in district) / population

        if not nx.is_connected(G.subgraph(district)):
            print(f"  {key} Skipping — disconnected.")
            continue

        root = nearest_node(G, district, mean_x, mean_y)
        district_set = set(district)
        assert root in district_set

        # --- Tree-based and Distance-based (unweighted shortest paths) ---
        pred = nx.predecessor(G, source=root)
        tree_feas = all((i == root or pred[i][0] in district_set) for i in district)
        dist_feas = all((i == root or any(j in district_set for j in pred[i])) for i in district)

        # --- DAG-based ---
        sp_length = nx.single_source_shortest_path_length(G, source=root)
        ordering = sorted(sp_length.items(), key=lambda item: item[1])
        position = {ordering[p][0]: p for p in range(len(ordering))}
        dag_feas = all(
            i == root or any(
                (j in district_set and position[j] < position[i])
                for j in G.neighbors(i)
            )
            for i in district
        )

        print(f"  {key}  tree={tree_feas}  dist={dist_feas}  dag={dag_feas}")

        # Tally results
        if tree_feas:
            counts["ttt"] += 1
        elif dist_feas:
            counts["ftt"] += 1
        elif dag_feas:
            counts["fft"] += 1
        else:
            counts["fff"] += 1

****************************************
Starting CA CD
****************************************

  nodes: 519723, edges: 1291913
  12  tree=False  dist=False  dag=False
  14  tree=False  dist=False  dag=False
  10  tree=False  dist=False  dag=False
  17  tree=False  dist=False  dag=False
  3  tree=False  dist=False  dag=False
  5  tree=False  dist=False  dag=False
  1  tree=False  dist=False  dag=False
  8  tree=False  dist=False  dag=False
  9  tree=False  dist=False  dag=False
  2  tree=False  dist=False  dag=False
  21  tree=False  dist=False  dag=False
  20  tree=False  dist=False  dag=False
  13  tree=False  dist=False  dag=False
  25  tree=False  dist=False  dag=False
  22  tree=False  dist=False  dag=False
  23  tree=False  dist=False  dag=False
  4  tree=False  dist=False  dag=False
  30  tree=False  dist=False  dag=False
  29  tree=False  dist=False  dag=False
  27  tree=False  dist=False  dag=False
  32  tree=False  dist=False  dag=False
  34  tree=False  dist=False  dag=Fal

C:\Users\buchanan\AppData\Local\Temp\ipykernel_4164\3484724055.py:22: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  csv_file = pd.read_csv(filepath + filename, skipinitialspace=True)


  unassigned = [317614, 196049, 99521, 323457, 86162, 40041, 352216, 300564, 79646, 242638, 31220, 73859, 190186, 34278, 124089, 210771, 279298, 288647, 236817, 155036, 273094, 184309, 145762, 334648, 177638, 44887, 213779, 314024, 320884, 2750, 319838, 24458, 16826, 66553, 230627, 218982, 183476, 245084, 161196, 267814, 69064, 369442, 139729, 157331, 184294, 47942, 341116, 264934, 348297, 257284, 279546, 210020, 297003, 221341, 31623, 99383, 366069, 158660, 191930, 331828, 172010, 280233, 143048, 94441, 128985, 16025, 331763, 221444, 146452, 59878, 360406, 157887]
  15  tree=False  dist=False  dag=False
  12  tree=False  dist=False  dag=False
  11  tree=False  dist=False  dag=False
  16  tree=False  dist=False  dag=False
  14  tree=False  dist=False  dag=False
  17  tree=False  dist=False  dag=False
  13  tree=False  dist=False  dag=False
  02 Skipping — disconnected.
  09  tree=False  dist=False  dag=False
  05  tree=False  dist=False  dag=False
  03  tree=False  dist=False  dag=Fals

C:\Users\buchanan\AppData\Local\Temp\ipykernel_4164\3484724055.py:22: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  csv_file = pd.read_csv(filepath + filename, skipinitialspace=True)


  unassigned = [317614, 196049, 99521, 323457, 86162, 40041, 352216, 300564, 79646, 242638, 31220, 73859, 190186, 34278, 124089, 210771, 279298, 288647, 236817, 155036, 273094, 184309, 145762, 334648, 177638, 44887, 213779, 314024, 320884, 2750, 319838, 24458, 16826, 66553, 230627, 218982, 183476, 245084, 161196, 267814, 69064, 369442, 139729, 157331, 184294, 47942, 341116, 264934, 348297, 257284, 279546, 210020, 297003, 221341, 31623, 99383, 366069, 158660, 191930, 331828, 172010, 280233, 143048, 94441, 128985, 16025, 331763, 221444, 146452, 59878, 360406, 157887]
  050 Skipping — disconnected.
  047  tree=False  dist=False  dag=False
  059 Skipping — disconnected.
  055  tree=False  dist=False  dag=False
  034  tree=False  dist=False  dag=False
  035  tree=False  dist=False  dag=False
  045 Skipping — disconnected.
  037  tree=False  dist=False  dag=False
  038  tree=False  dist=False  dag=False
  053  tree=False  dist=False  dag=False
  052  tree=False  dist=False  dag=False
  051  

C:\Users\buchanan\AppData\Local\Temp\ipykernel_4164\3484724055.py:22: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  csv_file = pd.read_csv(filepath + filename, skipinitialspace=True)


  unassigned = [317614, 196049, 99521, 323457, 86162, 40041, 352216, 300564, 79646, 242638, 31220, 73859, 190186, 34278, 124089, 210771, 279298, 288647, 236817, 155036, 273094, 184309, 145762, 334648, 177638, 44887, 213779, 314024, 320884, 2750, 319838, 24458, 16826, 66553, 230627, 218982, 183476, 245084, 161196, 267814, 69064, 369442, 139729, 157331, 184294, 47942, 341116, 264934, 348297, 257284, 279546, 210020, 297003, 221341, 31623, 99383, 366069, 158660, 191930, 331828, 172010, 280233, 143048, 94441, 128985, 16025, 331763, 221444, 146452, 59878, 360406, 157887]
  099  tree=False  dist=False  dag=False
  094  tree=False  dist=False  dag=False
  100  tree=False  dist=False  dag=False
  118  tree=False  dist=False  dag=False
  110  tree=False  dist=False  dag=False
  109  tree=False  dist=False  dag=False
  068  tree=False  dist=False  dag=False
  069  tree=False  dist=False  dag=False
  089 Skipping — disconnected.
  090  tree=True  dist=True  dag=True
  073  tree=False  dist=False  

In [7]:
# Summarize
total = sum(counts.values())
tree = counts["ttt"]
dist = counts["ftt"] + counts["ttt"]
dag = counts["fft"] + counts["ftt"] + counts["ttt"]

print(f"Counts: {counts}")
print(f"Total districts: {total}")
print()
print(f"Tree-based:     {tree}/{total} -> {round(100*tree/total, 2)}%")
print(f"Distance-based: {dist}/{total} -> {round(100*dist/total, 2)}%")
print(f"DAG-based:      {dag}/{total} -> {round(100*dag/total, 2)}%")

Counts: {'ttt': 161, 'ftt': 269, 'fft': 322, 'fff': 6345}
Total districts: 7097

Tree-based:     161/7097 -> 2.27%
Distance-based: 430/7097 -> 6.06%
DAG-based:      752/7097 -> 10.6%
